# Backend pool Load Balancing lab

![flow](./img/backend-pool-load-balancing.gif)

Playground to try the built-in load balancing [backend pool functionality of APIM](https://learn.microsoft.com/azure/api-management/backends?tabs=bicep) to a list of Azure OpenAI endpoints.

Notes:
- **This is a typical prioritized PTU with fallback consumption scenario**. The lab specifically showcases how a priority 1 (highest) backend is exhausted before gracefully falling back to two equally-weighted priority 2 backends.
- The backend pool uses round-robin by default.
- Priority and weight-based routing are supported and can be adjusted by modifying `priority` (the lower the number, the higher the priority) and `weight` variables in the `openai_resources` variable below.
- The `retry` API Management policy initiates a retry to an available backend if an HTTP 429 status code is encountered. This is transparent to the caller.

### ⚙️ Initialization

1. Get Az Cli authentication information
1. Get terraform outputs (prerequisite: run the [setup.ipynb](./setup.ipynb))

In [ ]:
import sys
sys.path.insert(1, './shared')  # add the shared directory to the Python path
import utils

terraform_dir = 'terraform'

output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

output = utils.run(f"terraform -chdir={terraform_dir} output -json", "Retrieved Terraform outputs", "Failed to retrieve Terraform outputs")

if output.success and output.json_data:
    apim_gateway_url = output.json_data['apim_gateway_url']['value']
    inference_api_path = output.json_data['load_balancing']['value']['inference_api_path']
    inference_api_key = output.json_data['load_balancing']['value']['inference_api_key']
    inference_api_version = output.json_data['load_balancing']['value']['inference_api_version']
    model_deployment_name = output.json_data['load_balancing']['value']['deployment_name']
    model_api_version = output.json_data['load_balancing']['value']['deployment_model']['version']

    utils.print_info(f"APIM Gateway URL: {apim_gateway_url}")
    utils.print_info(f"Inference API path: {inference_api_path}")
    utils.print_info(f"Inference API key: {inference_api_key}")
    utils.print_info(f"Inference API version: {inference_api_version}")
    utils.print_info(f"Model deployment name: {model_deployment_name}")
    utils.print_info(f"Model API version: {model_api_version}")

<a id='requests'></a>
### 🧪 Test the API using a direct HTTP call
Requests is an elegant and simple HTTP library for Python that will be used here to make raw API requests and inspect the responses. 

You will not see HTTP 429s returned as API Management's `retry` policy will select an available backend. If no backends are viable, an HTTP 503 will be returned.

Tip: Use the [tracing tool](../../tools/tracing.ipynb) to track the behavior of the backend pool.

In [ ]:
import json
import requests, time

runs = 20
sleep_time_ms = 100
url = f"{apim_gateway_url}/{inference_api_path}/openai/deployments/{model_deployment_name}/chat/completions?api-version={inference_api_version}"

utils.print_info(f"Endpoint: {url}")

messages = {"messages": [
    {"role": "system", "content": "You are a sarcastic, unhelpful assistant."},
    {"role": "user", "content": "Can you tell me the time, please?"}
]}
api_runs = []

# Initialize a session for connection pooling and set any default headers
session = requests.Session()
session.headers.update({'api-key': inference_api_key})

try:
    for i in range(runs):
        print(f"▶️ Run {i+1}/{runs}:")

        start_time = time.time()
        response = session.post(url, json = messages)
        response_time = time.time() - start_time
        print(f"⌚ {response_time:.2f} seconds")

        utils.print_response_code(response)

        if "x-ms-region" in response.headers:
            print(f"x-ms-region: \x1b[1;32m{response.headers.get("x-ms-region")}\x1b[0m") # this header is useful to determine the region of the backend that served the request
            api_runs.append((response_time, response.headers.get("x-ms-region")))

        if (response.status_code == 200):
            data = json.loads(response.text)
            print(f"Token usage: {json.dumps(dict(data.get("usage")), indent = 4)}\n")
            print(f"💬 {data.get("choices")[0].get("message").get("content")}\n")
        else:
            print(f"{response.text}\n")

        time.sleep(sleep_time_ms/1000)
finally:
    # Close the session to release the connection
    session.close()

<a id='plot'></a>
### 🔍 Analyze Load Balancing results

The priority 1 backend will be used until TPM exhaustion sets in, then distribution will occur near equally across the two priority 2 backends with 50/50 weights.  

Please note that the first request of the lab can take a bit longer and should be discounted in terms of duration.

In case the dependencies were not installed, you can install theme with the following command.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle as pltRectangle
import matplotlib as mpl

mpl.rcParams['figure.figsize'] = [15, 7]
df = pd.DataFrame(api_runs, columns = ['Response Time', 'Region'])
df['Run'] = range(1, len(df) + 1)

# Define a color map for each region
color_map = {'UK South': 'lightpink', 'France Central': 'lightblue', 'Sweden Central': 'lightyellow'}  # Add more regions and colors as needed

# Plot the dataframe with colored bars
ax = df.plot(kind = 'bar', x = 'Run', y = 'Response Time', color = [color_map.get(region, 'gray') for region in df['Region']], legend = False)

# Add legend
legend_labels = [pltRectangle((0, 0), 1, 1, color = color_map.get(region, 'gray')) for region in df['Region'].unique()]
ax.legend(legend_labels, df['Region'].unique())

plt.title('Load Balancing results')
plt.xlabel('Run #')
plt.ylabel('Response Time')
plt.xticks(rotation = 0)

average = df['Response Time'].mean()
plt.axhline(y = average, color = 'r', linestyle = '--', label = f'Average: {average:.2f}')

plt.show()

<a id='sdk'></a>
### 🧪 Test the API using the Azure OpenAI Python SDK

Repeat the same test using the Python SDK to ensure compatibility. Note that we do not know what region served the response; we only see that we obtained a response.

In [ ]:
import time
from openai import AzureOpenAI, APIStatusError

runs = 20
sleep_time_ms = 100

client = AzureOpenAI(
    azure_endpoint = f"{apim_gateway_url}/{inference_api_path}",
    api_key = inference_api_key,
    api_version = inference_api_version
)

for i in range(runs):
    print(f"▶️ Run {i+1}/{runs}:")

    start_time = time.time()
    try:
        raw_response = client.chat.completions.with_raw_response.create(
            model = model_deployment_name,
            messages = [
                {"role": "system", "content": "You are a sarcastic, unhelpful assistant."},
                {"role": "user", "content": "Can you tell me the time, please?"}
            ])
        response_time = time.time() - start_time

        print(f"⌚ {response_time:.2f} seconds")
        print(f"x-ms-region: \x1b[1;32m{raw_response.headers.get("x-ms-region")}\x1b[0m") # this header is useful to determine the region of the backend that served the request

        response = raw_response.parse()

        if response.usage:
            print(f"Token usage:\n   Total tokens: {response.usage.total_tokens}\n   Prompt tokens: {response.usage.prompt_tokens}\n   Completion tokens: {response.usage.completion_tokens}\n")

        print(f"💬 {response.choices[0].message.content}\n")
    except APIStatusError as e:
        response_time = time.time() - start_time
        print(f"⌚ {response_time:.2f} seconds")
        print(f"⚠️ Error: {e.status_code} - {e.message}\n")

    time.sleep(sleep_time_ms/1000)

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.